# Probability and Statistics for Machine Learning
Companion to the probability appendix. Each result in the notes, checked by calculation or simulation.

## 1. PMF, CDF, expectation and variance of songs per movie

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

songs = np.arange(8)
counts = np.array([10, 15, 25, 45, 20, 11, 9, 5])
pmf = counts / counts.sum()
cdf = pmf.cumsum()
print("PMF:", pmf.round(3), " sums to", pmf.sum())
print("P(X <= 4) =", cdf[4].round(3))
mean = (songs * pmf).sum()
var = (songs**2 * pmf).sum() - mean**2
print("E[X] =", round(mean, 4), " Var(X) =", round(var, 4), " sd =", round(np.sqrt(var), 4))

## 2. Expectation as a long-run average: simulate many movies

In [ ]:
rng = np.random.default_rng(0)
draws = rng.choice(songs, size=100_000, p=pmf)
print("average of 100,000 simulated movies:", draws.mean().round(4), " (expectation:", round(mean, 4), ")")
print("Var(2X + 5) =", np.var(2 * draws + 5).round(3), " vs 4 Var(X) =", (4 * var).round(3))

## 3. Distributions

In [ ]:
print("Binomial: P(7 heads in 10) =", stats.binom.pmf(7, 10, 0.5).round(4))
print("Poisson(3): P(0..5) =", stats.poisson.pmf(range(6), 3).round(4))
print("Normal: P(|Z| < 1) =", (stats.norm.cdf(1) - stats.norm.cdf(-1)).round(4),
      " P(|Z| < 1.96) =", (stats.norm.cdf(1.96) - stats.norm.cdf(-1.96)).round(4))
x = rng.normal(50, 10, size=10_000)
z = (x - x.mean()) / x.std()
print("z-scores: mean", z.mean().round(4), "sd", z.std().round(4))

## 4. Bayes: the positive test

In [ ]:
def posterior(prevalence, sensitivity=0.95, false_positive=0.05):
    p_pos = sensitivity * prevalence + false_positive * (1 - prevalence)
    return sensitivity * prevalence / p_pos

for prev in [0.001, 0.01, 0.1, 0.3]:
    print(f"prevalence {prev:>5}: P(disease | +) = {posterior(prev):.3f}")

## 5. Maximum likelihood for a coin

In [ ]:
p = np.linspace(0.01, 0.99, 99)
log_lik = 7 * np.log(p) + 3 * np.log(1 - p)
print("MLE from the grid:", p[log_lik.argmax()].round(2))
plt.plot(p, np.exp(log_lik)); plt.xlabel("p"); plt.ylabel("likelihood of 7 heads, 3 tails")
plt.axvline(0.7, color="red", ls="--")

## 6. Central limit theorem

In [ ]:
for n in [5, 30, 100]:
    means = np.array([rng.choice(songs, size=n, p=pmf).mean() for _ in range(5000)])
    print(f"n={n:>3}: sd of sample means {means.std():.4f}   theory sigma/sqrt(n) = {np.sqrt(var / n):.4f}")

## 7. Confidence interval and hypothesis test on the bank data

In [ ]:
bd = pd.read_csv("../../data/bd_train.csv")
high = (bd["Revenue.Grid"] == 1)
p_all, n_all = high.mean(), len(bd)
se = np.sqrt(p_all * (1 - p_all) / n_all)
print(f"overall rate {p_all:.4f}, 95% CI [{p_all - 1.96*se:.4f}, {p_all + 1.96*se:.4f}]")

rates = bd.assign(high=high).groupby("age_band")["high"].agg(["sum", "count", "mean"])
rates["z"] = (rates["mean"] - p_all) / np.sqrt(p_all * (1 - p_all) / rates["count"])
rates["p_value"] = 2 * (1 - stats.norm.cdf(rates["z"].abs()))
rates.round(4).sort_values("p_value")

## Try it yourself
1. $P(2 \le X \le 5)$ for the songs, using `cdf`.
2. Mean and variance of a fair die.
3. `posterior(0.10)`: how does it compare with 1%?
4. MLE for 2 heads in 10 tosses, by grid search.
5. CI for the 61-65 band (709 customers, 60 high-revenue).